# 07 - LangChain RAG Benchmark

This notebook evaluates simlar against other vector stores using LangChain as the integration layer. We measure how well each retriever finds the correct document for a given question — the core task in RAG before any LLM is involved.

**What we cover**
- Loading a question-answering dataset and building a ground-truth corpus
- Evaluating retrieval quality with Hit@1, Hit@k, and MRR
- Comparing simlar against Chroma, Qdrant, Milvus, and Pinecone via LangChain
- Measuring index build time and query latency

In [1]:
%pip install -q \
    sentence-transformers==5.6.0 datasets==5.0.0 \
    langchain-core==1.4.8 langchain-huggingface==1.2.2 langchain-chroma==1.1.0 langchain-community==0.4.2 langchain-deepseek==1.1.0 \
    qdrant-client==1.18.0 \
    pymilvus==3.0.0 pymilvus[milvus_lite]==3.0.0 \
    pinecone==9.1.0 \
    llama-index-core==0.14.23 llama-index-embeddings-huggingface==0.7.0 llama-index-llms-openai-like==0.7.2 \
    haystack-ai==2.31.0 numba==0.65.1

Note: you may need to restart the kernel to use updated packages.


## Environment variables

API keys for Pinecone and DeepSeek. Each prompt only appears if the key is not already set in the environment.

In [2]:
import os
from getpass import getpass


if not os.environ.get("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Pinecone API key: ")
if not os.environ.get("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass("DeepSeek API key: ")

## Dataset

We use [Kaggle](https://www.kaggle.com/datasets/ruhulaminsharif/squad-dataset) — a reading comprehension dataset where each question has a corresponding context passage that contains the answer. This structure maps naturally to RAG: the context passages become the corpus and the questions become the queries. For each query we know exactly which document should be retrieved, giving us a ground truth to evaluate against. Place the CSV file in the same directory as this notebook before running.

In [3]:
from datasets import load_dataset
from abc import ABC, abstractmethod


class Dataset(ABC):
    corpus: list[str]
    queries: list[str]

    @abstractmethod
    def evaluate(self, query: str, docs: list[str]) -> bool: ...

    @abstractmethod
    def rank(self, query: str, docs: list[str]) -> int | None: ...


class SquadDataset(Dataset):
    def __init__(self, corpus, queries, query_to_context):
        self.corpus = corpus
        self.queries = queries
        self._query_to_context = query_to_context

    @classmethod
    def from_hf(cls, n_questions=None, n_contexts=None):
        hf = load_dataset("rajpurkar/squad", split="train")
        seen, corpus, queries, query_to_context = set(), [], [], {}
        for row in hf:
            if n_questions is not None and len(queries) == n_questions:
                break
            ctx = row["context"]
            if ctx not in seen:
                if n_contexts is not None and len(corpus) >= n_contexts:
                    continue
                seen.add(ctx)
                corpus.append(ctx)
            queries.append(row["question"])
            query_to_context[row["question"]] = ctx
        return cls(corpus=corpus, queries=queries, query_to_context=query_to_context)

    def evaluate(self, query, docs):
        return self._query_to_context.get(query) in docs

    def rank(self, query: str, docs: list[str]) -> int | None:
        correct = self._query_to_context.get(query)
        for i, doc in enumerate(docs):
            if doc == correct:
                return i
        return None


class MsMarcoDataset(Dataset):
    def __init__(self, corpus, queries, query_to_selected):
        self.corpus = corpus
        self.queries = queries
        self._query_to_selected = query_to_selected

    @classmethod
    def from_hf(cls, n_queries=None, n_passages=None):
        hf = load_dataset("microsoft/ms_marco", "v2.1", split="train")
        seen, passages, queries, query_to_selected = set(), [], [], {}

        for row in hf:
            if n_passages is None or len(passages) < n_passages:
                for text in row["passages"]["passage_text"]:
                    if text not in seen:
                        seen.add(text)
                        passages.append(text)

            if n_queries is None or len(queries) < n_queries:
                selected = {t for t, s in zip(row["passages"]["passage_text"], row["passages"]["is_selected"]) if s == 1}
                if selected and selected & set(passages):
                    queries.append(row["query"])
                    query_to_selected[row["query"]] = selected & set(passages)

            if (n_passages is None or len(passages) >= n_passages) and \
            (n_queries is None or len(queries) >= n_queries):
                break

        return cls(corpus=passages, queries=queries, query_to_selected=query_to_selected)

    def evaluate(self, query, docs):
        return bool(self._query_to_selected.get(query, set()) & set(docs))

    def rank(self, query, docs):
        selected = self._query_to_selected.get(query, set())
        for i, doc in enumerate(docs):
            if doc in selected:
                return i
        return None    

/home/rmora/miniconda3/envs/simlar_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Suits

Each suit wraps a vector store and exposes a uniform LangChain interface: given a query string, return the top-k most relevant documents from the corpus. We compare simlar against Chroma, Qdrant, Milvus, and Pinecone.

All suits use the same embedding model (`all-MiniLM-L6-v2`) so that differences in retrieval quality reflect the underlying index, not the embeddings.

In [4]:
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.retrievers import BaseRetriever as LangchainBaseRetriever

### simlar

simlar exposes a hybrid index that combines keyword (BM25) and semantic (vector) search. It integrates natively with LangChain via `SimlarVectorStore`.

In [5]:

def simlar(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    from simlar.integrations.langchain.langchain_retriever import SimlarRetriever
    from simlar.integrations.langchain.simlar_vector_store import SimlarVectorStore
    SimlarRetriever.model_rebuild()
    store = SimlarVectorStore.from_texts(
        texts=dataset.corpus,
        embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    )
    return SimlarRetriever(vector_store=store, k=k)

### Chroma

[Chroma](https://www.trychroma.com/) is an open-source embedding database designed for AI applications. It stores vectors in memory or on disk and supports similarity search out of the box. In this benchmark we use it in in-memory mode so there is no persistence overhead between runs.

In [6]:
from langchain_chroma import Chroma

def chroma(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    store = Chroma.from_texts(
        texts=dataset.corpus,
        embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    )
    return store.as_retriever(search_kwargs={"k": k})

### Qdrant

[Qdrant](https://qdrant.tech/) is a vector search engine built in Rust, optimized for high-performance similarity search. It supports filtering, payloads, and multiple distance metrics. Here we run it in in-memory mode (`":memory:"`) to keep setup simple and avoid disk I/O in the benchmark.

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def qdrant(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    _model = SentenceTransformer("all-MiniLM-L6-v2")
    _corpus = dataset.corpus
    vectors = _model.encode(_corpus, normalize_embeddings=True)
    _client = QdrantClient(":memory:")
    _client.create_collection(
        collection_name="corpus",
        vectors_config=VectorParams(size=vectors.shape[1], distance=Distance.COSINE),
    )
    _client.upsert(
        collection_name="corpus",
        points=[PointStruct(id=i, vector=vectors[i].tolist(), payload={"text": text})
                for i, text in enumerate(_corpus)],
    )

    class _R(LangchainBaseRetriever):
        def _get_relevant_documents(self, query, *, run_manager=None):
            qv = _model.encode([query], normalize_embeddings=True)[0].tolist()
            return [Document(page_content=r.payload["text"])
                    for r in _client.query_points(collection_name="corpus", query=qv, limit=k).points]

    return _R()

### Milvus

[Milvus](https://milvus.io/) is a distributed vector database built for large-scale similarity search. We use [Milvus Lite](https://milvus.io/docs/milvus_lite.md) — a lightweight version that runs locally as a file-based database, with no server required. Note that only one process can open the database file at a time.

In [8]:
from pymilvus import MilvusClient

def milvus(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    _model = SentenceTransformer("all-MiniLM-L6-v2")
    _corpus = dataset.corpus
    vectors = _model.encode(_corpus, normalize_embeddings=True)
    _client = MilvusClient("/tmp/milvus_langchain.db")
    if _client.has_collection("corpus"):
        _client.drop_collection("corpus")
    _client.create_collection(collection_name="corpus", dimension=vectors.shape[1])
    _client.insert(
        collection_name="corpus",
        data=[{"id": i, "vector": vectors[i].tolist(), "text": text}
              for i, text in enumerate(_corpus)],
    )

    class _R(LangchainBaseRetriever):
        def _get_relevant_documents(self, query, *, run_manager=None):
            qv = _model.encode([query], normalize_embeddings=True)[0].tolist()
            results = _client.search(collection_name="corpus", data=[qv], limit=k, output_fields=["text"])
            return [Document(page_content=r["entity"]["text"]) for r in results[0]]

    return _R()

### Pinecone

[Pinecone](https://www.pinecone.io/) is a managed vector database — unlike the others, it runs as an external cloud service. This means build time includes network latency for uploading vectors, and query time includes a round-trip to Pinecone's servers. A `PINECONE_API_KEY` environment variable is required.

In [9]:
def pinecone(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    from sentence_transformers import SentenceTransformer
    from pinecone import Pinecone, ServerlessSpec
    _model = SentenceTransformer("all-MiniLM-L6-v2")
    _corpus = dataset.corpus
    vectors = _model.encode(_corpus, normalize_embeddings=True)
    pc = Pinecone()
    if "rag-corpus" not in [i.name for i in pc.list_indexes()]:
        pc.create_index(
            name="rag-corpus",
            dimension=vectors.shape[1],
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
    _index = pc.Index("rag-corpus")
    batch_size = 100
    records = [{"id": str(i), "values": vectors[i].tolist(), "metadata": {"text": text}}
               for i, text in enumerate(_corpus)]
    for i in range(0, len(records), batch_size):
        _index.upsert(vectors=records[i:i + batch_size])

    class _R(LangchainBaseRetriever):
        def _get_relevant_documents(self, query, *, run_manager=None):
            qv = _model.encode([query], normalize_embeddings=True)[0].tolist()
            results = _index.query(vector=qv, top_k=k, include_metadata=True)
            return [Document(page_content=m["metadata"]["text"]) for m in results["matches"]]

    return _R()

### FAISS

[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search) is a library for efficient similarity search over dense vectors. It runs entirely in memory with no server required, using optimized CPU/GPU kernels. Unlike the other stores in this benchmark, FAISS is pure vector search — no keyword component — making it a useful baseline for semantic-only retrieval.

In [10]:
from langchain_community.vectorstores import FAISS as FaissStore

def faiss(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    store = FaissStore.from_texts(
        texts=dataset.corpus,
        embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    )
    return store.as_retriever(search_kwargs={"k": k})

/tmp/ipykernel_1738940/3350315975.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as FaissStore


### turbovec

[turbovec](https://github.com/RyanCodrai/turbovec) is a vector search library implemented in Rust with Python bindings, built on Google Research's TurboQuant algorithm. It uses 2–4 bit quantization to achieve up to 16x compression — a 10M document corpus that takes 31GB as float32 fits in 4GB. Hand-optimized SIMD kernels (AVX-512, NEON) make it faster than FAISS on ARM. Like FAISS, it is pure vector search with no keyword component.

In [11]:
from turbovec.langchain import TurboQuantVectorStore

def turbovec(dataset: Dataset, k: int = 3) -> LangchainBaseRetriever:
    store = TurboQuantVectorStore.from_texts(
        texts=dataset.corpus,
        embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
        bit_width=4,
    )
    return store.as_retriever(search_kwargs={"k": k})

## RAG

`LangchainRAG` wraps a LangChain retriever and exposes two methods:
- `retrieve(query)` — returns the top-k documents as strings, no LLM involved. Used for the benchmark.
- `ask(query)` — runs the full RAG pipeline with an LLM and returns an answer.

In [12]:
from dataclasses import dataclass
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_deepseek import ChatDeepSeek


@dataclass
class AskResult:
    question: str
    answer: str | None
    docs: list[str]
    hit: bool


class RAG:
    def retrieve(self, query: str) -> list[str]: ...
    def ask(self, query: str) -> AskResult: ...


class LangchainRAG(RAG):
    def __init__(self, dataset: Dataset, retriever: LangchainBaseRetriever):
        self._dataset = dataset
        self._retriever = retriever
        prompt = ChatPromptTemplate.from_template(
            "Answer using only the context below. Be concise.\n\n"
            "Context:\n{context}\n\nQuestion: {question}"
        )
        self._chain = (
            RunnableParallel(docs=retriever, question=RunnablePassthrough())
            | RunnablePassthrough.assign(context=lambda x: "\n\n".join(d.page_content for d in x["docs"]))
            | RunnableParallel(
                answer=prompt | ChatDeepSeek(model="deepseek-chat") | StrOutputParser(),
                docs=lambda x: x["docs"],
            )
        )

    def retrieve(self, query: str) -> list[str]:
        return [d.page_content for d in self._retriever.invoke(query)]

    def ask(self, query: str) -> AskResult:
        result = self._chain.invoke(query)
        docs = [d.page_content for d in result["docs"]]
        return AskResult(question=query, answer=result["answer"], docs=docs,
                         hit=self._dataset.evaluate(query, docs))

## Benchmark

`test_retrievers` runs the full benchmark in two phases:

1. **Build** — each retriever indexes the corpus and the elapsed time is recorded.
2. **Query** — every question in the dataset is passed to `rag.retrieve()` and the result is evaluated against the ground truth.

### Metrics

- **Hit@1** — fraction of queries where the correct document is ranked first.
- **Hit@k** — fraction of queries where the correct document appears in the top k results.
- **MRR** (Mean Reciprocal Rank) — average of `1/rank` for the correct document. Penalizes results that are correct but ranked lower.

In [13]:
import io
import time
import contextlib
import pandas as pd
from typing import Callable
import logging

logging.getLogger("pymilvus").setLevel(logging.CRITICAL)

@contextlib.contextmanager
def _suppress():
    devnull = os.open(os.devnull, os.O_WRONLY)
    old_stdout = os.dup(1)
    old_stderr = os.dup(2)
    os.dup2(devnull, 1)
    os.dup2(devnull, 2)
    os.close(devnull)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            with contextlib.redirect_stderr(io.StringIO()):
                yield
    finally:
        os.dup2(old_stdout, 1)
        os.dup2(old_stderr, 2)
        os.close(old_stdout)
        os.close(old_stderr)


def test_retrievers(dataset: SquadDataset, out: str, k: int = 3) -> None:
    print(f"Dataset: {len(dataset.corpus)} documents & {len(dataset.queries)} queries")

    entries: list[tuple[str, Callable[[], object], Callable[[object], RAG]]] = [
        ("simlar", lambda: simlar(dataset, k), lambda r: LangchainRAG(dataset, r)),
        ("chroma", lambda: chroma(dataset, k), lambda r: LangchainRAG(dataset, r)),
        ("qdrant", lambda: qdrant(dataset, k), lambda r: LangchainRAG(dataset, r)),
        ("milvus", lambda: milvus(dataset, k), lambda r: LangchainRAG(dataset, r)),
        # ("pinecone", lambda: pinecone(dataset, k), lambda r: LangchainRAG(dataset, r)),
        ("faiss", lambda: faiss(dataset, k), lambda r: LangchainRAG(dataset, r)),
        ("turbovec", lambda: turbovec(dataset, k), lambda r: LangchainRAG(dataset, r)),
    ]

    # Build phase
    rags: list[dict] = []
    for name, retriever_factory, rag_factory in entries:
        print(f"Building {name}...")
        try:
            with _suppress():
                t0 = time.perf_counter()
                retriever = retriever_factory()
                build_ms = (time.perf_counter() - t0) * 1000
                rag = rag_factory(retriever)
            rags.append({"name": name, "rag": rag, "build_ms": build_ms})
        except Exception as e:
            print(f"  skipped: {e}")

    # Query phase
    n = len(dataset.queries)
    for entry in rags:
        name, rag = entry["name"], entry["rag"]
        hit1 = hitk = mrr = 0.0
        errors = 0
        t0 = time.perf_counter()
        for i, q in enumerate(dataset.queries, 1):
            try:
                with _suppress():
                    docs = rag.retrieve(q)
                    r = dataset.rank(q, docs)
                if r is not None:
                    if r == 0: hit1 += 1
                    hitk += 1
                    mrr += 1 / (r + 1)
            except Exception as e:
                errors += 1
                with open("errors.txt", "a") as f:
                    f.write(f"[{name}] query={repr(q)} error={e}\n")
            if i % 100 == 0 or i == n:
                print(f"\r  {name}: {i}/{n} queries{f' ({errors} errors)' if errors else ''}", end="", flush=True)
        print()
        entry["query_ms"] = (time.perf_counter() - t0) * 1000 / n
        entry["hit1"] = hit1 / n
        entry["hitk"] = hitk / n
        entry["mrr"]  = mrr  / n

    # Results
    hitk_col = f"Hit@{k}"
    rows = [
        {
            "Retriever": e["name"],
            "Hit@1":     e["hit1"],
            hitk_col:    e["hitk"],
            "MRR":       e["mrr"],
            "Build(ms)": e["build_ms"],
            "Query(ms)": e["query_ms"],
        }
        for e in rags
    ]
    df = pd.DataFrame(rows).set_index("Retriever")

    display(df.style.format({
        "Hit@1":     "{:.1%}",
        hitk_col:    "{:.1%}",
        "MRR":       "{:.3f}",
        "Build(ms)": "{:.0f}",
        "Query(ms)": "{:.1f}",
    }))

    # Save to CSV
    df.reset_index().rename(columns={
        "Retriever": "name", "Hit@1": "hit1", hitk_col: "hitk",
        "MRR": "mrr", "Build(ms)": "build_ms", "Query(ms)": "query_ms",
    }).to_csv(out, index=False)
    print(f"\nSaved to {out}")

In [14]:
# dataset = SquadDataset.from_hf(n_questions=100)
dataset = MsMarcoDataset.from_hf(n_passages=500_000, n_queries=20_000)
test_retrievers(dataset, "langchain_benchmark_results.csv", k=10)

Dataset: 500003 documents & 20000 queries
Building simlar...
Building chroma...
Building qdrant...
Building milvus...
  skipped: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.RESOURCE_EXHAUSTED
	details = "SERVER: Received message larger than max (943897710 vs. 268435456)"
	debug_error_string = "RESOURCE_EXHAUSTED:SERVER: Received message larger than max (943897710 vs. 268435456)"
>
Building faiss...
Building turbovec...
  simlar: 20000/20000 queries (1 errors)
  chroma: 20000/20000 queries
  qdrant: 20000/20000 queries
  faiss: 20000/20000 queries
  turbovec: 20000/20000 queries


,Hit@1,Hit@10,MRR,Build(ms),Query(ms)
Retriever,,,,,
simlar,29.7%,85.6%,0.472,133204,8.5
chroma,33.6%,86.2%,0.505,233657,2.8
qdrant,33.8%,86.8%,0.509,236668,417.2
faiss,33.9%,86.8%,0.509,136133,27.9
turbovec,33.9%,86.8%,0.509,130317,12.0



Saved to langchain_benchmark_results.csv
